In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
from retain_dss.data.schema import MATERIAL_INPUTS, EconomicParams
from retain_dss.models.trainer import load_models
from retain_dss.optimizer.genetic import run_nsga2
from retain_dss.optimizer.route_selector import select_best_route

In [ ]:
mat_inputs = {k: (v["range"][0]+v["range"][1])/2 for k, v in MATERIAL_INPUTS.items()}
models = {r: load_models(r) for r in ["route1","route2","route3"]}
elec_prices = np.linspace(0.05, 0.40, 8)
margins = []
recommended = []
for ep in elec_prices:
    prices = EconomicParams(electricity_price=ep)
    results = {r: run_nsga2(mat_inputs, models[r], r, prices, 50, 50, 42) for r in ["route1","route2","route3"]}
    sel = select_best_route(results)
    margins.append(-sel["best_objectives_raw"][4])
    recommended.append(sel["recommended_route"])
    print(f"  elec={ep:.2f}: best margin = {margins[-1]:.1f} €/t, route = {recommended[-1]}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=elec_prices, y=margins, mode="lines+markers",
    marker=dict(color="#6366f1", size=8), line=dict(width=2)))
fig.update_layout(xaxis_title="Electricity price (€/kWh_el)",
                  yaxis_title="Best net margin (€/t)",
                  template="plotly_dark", title="Sensitivity to electricity price")
fig.show()

In [ ]:
pvc_prices = np.linspace(200, 700, 8)
margins_pvc = []
for pp in pvc_prices:
    prices = EconomicParams(price_pvc_recycled=pp)
    results = {r: run_nsga2(mat_inputs, models[r], r, prices, 50, 50, 42) for r in ["route1","route2","route3"]}
    sel = select_best_route(results)
    margins_pvc.append(-sel["best_objectives_raw"][4])
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=pvc_prices, y=margins_pvc, mode="lines+markers",
    marker=dict(color="#f59e0b", size=8), line=dict(width=2)))
fig2.update_layout(xaxis_title="PVC recycled price (€/t)", yaxis_title="Best net margin (€/t)",
                   template="plotly_dark", title="Sensitivity to PVC market price")
fig2.show()